In [ ]:
!date

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats
from concurrent.futures import ProcessPoolExecutor
from statsmodels.stats.multitest import multipletests

In [ ]:
projdir = '/u/project/cluo/terencew/claude/project_ideas/asm_lr_hprc2'
figdir = f'{projdir}/figures/asm'
outdir = f'{projdir}/results/qc/data/qc16'
!mkdir -p {outdir}
chrom = 'chr20'
def lam(p):
    return np.median(stats.chi2.isf(np.clip(p, 1e-300, 1), 1)) / 0.4549

### per-donor diagnostics from the real chr20 calls

In [ ]:
freq = pd.read_csv(f'{projdir}/results/meth_bins/domain_frequency_10kb.tsv.gz', sep='\t', usecols=['chrom', 'bin_start', 'class_pmd'])
freq = freq[freq['chrom'] == chrom]

def donor_qc(f):
    d = pd.read_csv(f, sep='\t', usecols=['sample', 'set', 'start', 'n_reads1', 'n_reads2', 'mu1', 'mu2', 'delta', 'pval', 'method'])
    c = d[d['set'] == 'cpg'].copy()
    if not len(c):
        return None
    c['bin_start'] = (c['start'] // 10000) * 10000
    c = c.merge(freq, on='bin_start', how='left')
    inp = c['class_pmd'].isin(['constitutive', 'common'])
    nev = c['class_pmd'] == 'never'
    z = d[d['set'] == 'zink']
    return {'sample': c['sample'].iloc[0], 'n_tested': len(c), 'lambda_real': lam(c['pval']),
            'lambda_in_domain': lam(c['pval'][inp]) if inp.sum() > 100 else np.nan,
            'lambda_never': lam(c['pval'][nev]) if nev.sum() > 100 else np.nan,
            'frac_p05': (c['pval'] < 0.05).mean(), 'frac_p1e4': (c['pval'] < 1e-4).mean(),
            'frac_delta_pos': (c['delta'] > 0).mean(), 'mean_delta': c['delta'].mean(),
            'median_absdelta': c['delta'].abs().median(), 'median_reads': np.median(c['n_reads1'] + c['n_reads2']),
            'read_imbalance': np.median((c['n_reads1'] - c['n_reads2']).abs() / (c['n_reads1'] + c['n_reads2'])),
            'frac_separation': (c['method'] == 'exact_separation').mean(),
            'zink_n': len(z), 'zink_frac_p1e4': (z['pval'] < 1e-4).mean() if len(z) else np.nan,
            'zink_median_absdelta': z['delta'].abs().median() if len(z) else np.nan}

files = sorted(glob.glob(f'{projdir}/results/asm/calls/*_{chrom}.asm.tsv.gz'))
with ProcessPoolExecutor(10) as ex:
    q = pd.DataFrame([r for r in ex.map(donor_qc, files) if r])
q.shape

In [ ]:
q.head()

### add the empirical null (C06a: split one haplotype in half), covariates and genome-wide yield

In [ ]:
nulls = sorted(glob.glob(f'{projdir}/results/asm/null/*_{chrom}.null_summary.tsv'))
nl = pd.concat([pd.read_csv(f, sep='\t') for f in nulls]) if nulls else pd.DataFrame()
if len(nl):
    nl = nl.groupby('sample').agg(lambda_null=('lambda_null', 'mean'), null_frac_p05=('frac_p05', 'mean'),
                                  null_mean_absdelta=('mean_absdelta', 'mean'),
                                  median_read_sd=('median_read_sd', 'mean')).reset_index()
print(len(nl), 'donors with empirical null so far')

In [ ]:
chem = pd.read_csv(f'{projdir}/results/qc/data/supp_seq_qc.csv').set_index('sample_id')['sequencing_chemistry_ont']
sp = pd.read_csv(f'{projdir}/tsv/meta/hprc2_sample_manifest.tsv', sep='\t').set_index('sample_id')['superpopulation']
dm = pd.read_csv(f'{projdir}/results/qc/data/qc13/donor_pmd_expansion_metrics.tsv', sep='\t')[
    ['sample', 'depth_constitutive', 'global_meth', 'mcg_never']]
ds = pd.read_csv(f'{projdir}/results/asm/genome/donor_summary.tsv', sep='\t')[
    ['sample', 'n_tested', 'n_asm', 'frac_asm', 'frac_reads_het', 'frac_reads_kept', 'median_reads']].rename(
    columns={'n_tested': 'gw_n_tested', 'n_asm': 'gw_n_asm', 'median_reads': 'gw_median_reads'})
xist = pd.read_csv(f'{projdir}/results/qc/data/xist_promoter_skew.tsv', sep='\t')[['sample', 'skew']].rename(columns={'skew': 'xist_skew'})
q['chemistry'] = q['sample'].map(chem); q['superpop'] = q['sample'].map(sp)
q = q.merge(dm, on='sample', how='left').merge(ds, on='sample', how='left').merge(xist, on='sample', how='left')
if len(nl):
    q = q.merge(nl, on='sample', how='left')
    q['excess_lambda'] = q['lambda_real'] / q['lambda_null']
q.describe().loc[['mean', 'std', 'min', '50%', 'max']].round(3).T

### is the test itself miscalibrated, or are the haplotype differences real?

In [ ]:
if 'lambda_null' in q:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    sns.histplot(q['lambda_null'].dropna(), bins=30, ax=axes[0]); axes[0].axvline(1, c='r', ls='--')
    axes[0].set_xlabel('lambda under within-haplotype null (1 = calibrated)')
    sns.scatterplot(data=q, x='lambda_null', y='lambda_real', hue='chemistry', s=16, ax=axes[1])
    axes[1].axhline(1, c='gray', ls=':'); axes[1].axvline(1, c='gray', ls=':'); axes[1].set_yscale('log')
    sns.scatterplot(data=q, x='depth_constitutive', y='lambda_real', hue='xist_skew', s=18, ax=axes[2])
    axes[2].set_yscale('log')
    plt.tight_layout(); plt.savefig(f'{figdir}/qc16_calibration.png', dpi=150)
    print(q[['lambda_null', 'lambda_real', 'excess_lambda']].describe().round(3).to_string())

### is the inflation confined to PMDs?

In [ ]:
sub = q.dropna(subset=['lambda_in_domain', 'lambda_never'])
print('median lambda in-domain %.2f vs never-PMD %.2f' % (sub['lambda_in_domain'].median(), sub['lambda_never'].median()))
print('donors where never-PMD lambda > 1.5:', (sub['lambda_never'] > 1.5).sum(), '/', len(sub))
fig, ax = plt.subplots(figsize=(4.5, 4.2))
sns.scatterplot(data=sub, x='lambda_never', y='lambda_in_domain', hue='depth_constitutive', s=18, ax=ax)
lim = [0.5, sub[['lambda_never', 'lambda_in_domain']].max().max() * 1.05]
ax.plot(lim, lim, ls='--', c='gray'); ax.set_xscale('log'); ax.set_yscale('log')
plt.tight_layout(); plt.savefig(f'{figdir}/qc16_inflation_in_vs_out_domains.png', dpi=150)

### what predicts the donor-level inflation

In [ ]:
d = q.dropna(subset=['lambda_real', 'depth_constitutive'])
terms = ['depth_constitutive', 'xist_skew', 'frac_reads_het', 'median_reads', 'read_imbalance', 'global_meth', 'mcg_never']
rows = []
for t in terms:
    x = d.dropna(subset=[t])
    f0 = smf.ols('np.log(lambda_real) ~ C(chemistry) + C(superpop)', x).fit()
    f1 = smf.ols(f'np.log(lambda_real) ~ C(chemistry) + C(superpop) + {t}', x).fit()
    rows.append({'covariate': t, 'n': len(x), 'partial_r2': (f0.ssr - f1.ssr) / f0.ssr, 'p': f1.pvalues[t]})
pd.DataFrame(rows).round(4)

In [ ]:
full = smf.ols('np.log(lambda_real) ~ depth_constitutive + xist_skew + C(chemistry) + C(superpop) + median_reads', d.dropna(subset=['xist_skew'])).fit()
print('R2 %.3f' % full.rsquared)
pd.DataFrame({'coef': full.params, 'p': full.pvalues}).round(4)

### positive control: Zink imprinted DMRs vs inflation

In [ ]:
zk = pd.read_csv(f'{projdir}/results/asm/genome/zink_control.tsv', sep='\t')
zd = zk.groupby('sample').agg(zink_det=('asm', 'mean'), zink_absd=('delta', lambda x: x.abs().mean())).reset_index()
qz = q.merge(zd, on='sample', how='left')
print('corr(zink detection, lambda_real) = %.3f' % qz[['zink_det', 'lambda_real']].corr().iloc[0, 1])
print('corr(zink |delta|, lambda_real)   = %.3f' % qz[['zink_absd', 'lambda_real']].corr().iloc[0, 1])
print(qz.groupby(pd.qcut(qz['lambda_real'], 4))[['zink_det', 'zink_absd', 'frac_asm']].median().round(3).to_string())

### genomic-control recalibration: how many ASM calls survive per donor

In [ ]:
def recal(f):
    d = pd.read_csv(f, sep='\t', usecols=['sample', 'set', 'pval', 'delta'])
    c = d[d['set'] == 'cpg']
    if len(c) < 1000:
        return None
    l = lam(c['pval'])
    chi2 = stats.chi2.isf(np.clip(c['pval'], 1e-300, 1), 1) / max(l, 1.0)
    p_gc = stats.chi2.sf(chi2, 1)
    q_raw = multipletests(c['pval'], method='fdr_bh')[1]
    q_gc = multipletests(p_gc, method='fdr_bh')[1]
    big = c['delta'].abs() >= 0.2
    return {'sample': c['sample'].iloc[0], 'lambda': l,
            'n_sig_raw': int(((q_raw < 0.05) & big).sum()), 'n_sig_gc': int(((q_gc < 0.05) & big).sum())}

with ProcessPoolExecutor(10) as ex:
    rc = pd.DataFrame([r for r in ex.map(recal, files) if r])
rc['ratio'] = rc['n_sig_gc'] / rc['n_sig_raw'].replace(0, np.nan)
print(rc[['lambda', 'n_sig_raw', 'n_sig_gc', 'ratio']].describe().round(3).to_string())
rc.sort_values('lambda', ascending=False).head(10).round(3)

### donor QC flags

In [ ]:
qq = q.merge(rc[['sample', 'n_sig_raw', 'n_sig_gc']], on='sample', how='left')
qq['flag_inflated'] = qq['lambda_real'] > 2
qq['flag_null_bad'] = (qq['lambda_null'] > 1.2) if 'lambda_null' in qq else False
qq['flag_low_cov'] = qq['median_reads'] < 20
qq['flag_sign_bias'] = (qq['frac_delta_pos'] - 0.5).abs() > 0.05
qq['flag_any'] = qq[['flag_inflated', 'flag_null_bad', 'flag_low_cov', 'flag_sign_bias']].any(axis=1)
print(qq[['flag_inflated', 'flag_null_bad', 'flag_low_cov', 'flag_sign_bias', 'flag_any']].sum().to_string())
qq.sort_values('lambda_real', ascending=False).head(12)[
    ['sample', 'superpop', 'chemistry', 'lambda_real', 'lambda_null' if 'lambda_null' in qq else 'frac_p05',
     'depth_constitutive', 'xist_skew', 'frac_asm', 'n_sig_raw', 'n_sig_gc']].round(3)

In [ ]:
qq.to_csv(f'{outdir}/asm_donor_qc_{chrom}.tsv', sep='\t', index=False)
rc.to_csv(f'{outdir}/asm_genomic_control_{chrom}.tsv', sep='\t', index=False)

In [ ]:
!date